In [1]:
#Import the necessary packages.
import os

#Adjust the settings according to your GPU specifications.
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ['DGLBACKEND'] = 'pytorch'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from graph_constructor import GraphDataset, collate_fn
from preprocess_complex import generate_pocket,generate_complex
from MSIGN import MSIGN
from utils import cal_dist, area_triangle, angle
import multiprocessing
from itertools import repeat
import networkx as nx
import dgl
import time
import numpy as np
np.set_printoptions(threshold=np.inf)
import pandas as pd
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from scipy.spatial import distance_matrix

from config.config_dict import *
from log.train_logger import *
from utils import *

import csv
import pickle
from tqdm import tqdm
import pymol
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

import traceback
import warnings

warnings.filterwarnings('ignore')

In [2]:
#First, you should prepare the ligand file (in pdb format) after docking with 5xr8.pdb. The receptor should be fixed using the 5xr8.pdb file provided in this project.
#Secondly, you can perform data preprocessing according to this step, which is expected to generate Pocket_5A.pdb, .rdkit, and .dgl files.
# .rdkit

distance = 5
input_ligand_format = 'pdb'
data_root = './data'
data_dir = os.path.join(data_root, 'out_pdb')
#You can prepare your own CSV file for testing; 
#please refer to molecule.csv for the specific format.
#data_df = pd.read_csv(os.path.join(data_root, 'molecule.csv'))
data_df = pd.read_csv(os.path.join(data_root, 'toy_test.csv'))
generate_pocket(data_dir=data_dir, distance=distance)
generate_complex(data_dir, data_df, distance=distance, input_ligand_format=input_ligand_format)



100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 301.00it/s]


In [3]:
# .dgl

data_root = './data'
data_dir = os.path.join(data_root, 'out_pdb')
data_df = pd.read_csv(os.path.join(data_root, 'toy_test.csv'))
data_set = GraphDataset(data_dir, data_df, graph_type='Atom_Graph2', dis_threshold=5., create=True)
data_loader = DataLoader(data_set, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=1)

Generate complex graph...


In [4]:
#Finally, these files were used for prediction.
device = torch.device('cuda')

for i,row in data_df.iterrows():
    molecule_dir = os.path.join(data_root, 'out_pdb/{}'.format(str(row['id'])))
    pred_dir = os.path.join(molecule_dir, 'Atom_Graph2-{}.dgl'.format(str(row['id'])))
    graph_data, label = torch.load(pred_dir)
    gd = graph_data.to(device)
    model = MSIGN(node_feat_size=35, edge_feat_size=17, hidden_feat_size=256, layer_num=3).to(device)
    model.load_state_dict(torch.load("./model/full_finetune_model.pt"))
    model.eval()
    pred_p, pred_c = model(gd)
    pred = (pred_p + pred_c) / 2
    print("{}:Prediction:%.4f".format(str(row['id'])) % pred)

molecule_997:Prediction:4.0276
molecule_998:Prediction:5.2760
molecule_999:Prediction:5.9723
